# 01 - TF-IDF + Cosine Similarity

This notebook implements the first semantic similarity method **TF-IDF + cosine similarity**.

TF-IDF is a useful baseline, simple, fast, and interpretable. However, it mainly measures lexical overlap, so it struggles when two sentences use different words to express the same meaning.

In [231]:
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np
import pandas as pd
import math

from scipy.stats import pearsonr, spearmanr


## Loading Semantic Textual Similarity Benchmark (STS-B) Data

In [2]:
from datasets import load_dataset

dataset = load_dataset("sentence-transformers/stsb")

In [175]:
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1379
    })
})

In [176]:
print(dataset["train"][0])

{'sentence1': 'A plane is taking off.', 'sentence2': 'An air plane is taking off.', 'score': 1.0}


In [177]:
print(dataset["train"][1]["sentence1"])

A man is playing a large flute.


In [178]:
dataset["train"][:5]

{'sentence1': ['A plane is taking off.',
  'A man is playing a large flute.',
  'A man is spreading shreded cheese on a pizza.',
  'Three men are playing chess.',
  'A man is playing the cello.'],
 'sentence2': ['An air plane is taking off.',
  'A man is playing a flute.',
  'A man is spreading shredded cheese on an uncooked pizza.',
  'Two men are playing chess.',
  'A man seated is playing the cello.'],
 'score': [1.0, 0.76, 0.76, 0.52, 0.85]}

## Manual TF-IDF implementation

In [ ]:
# Tokenized all sentences in data
tokenized_sents = []

for sample in dataset["train"]:
    tokenized_sents.append([token.lower() for token in word_tokenize(sample["sentence1"])])
    tokenized_sents.append([token.lower() for token in word_tokenize(sample["sentence2"])])

In [181]:
tokenized_sents[:5]

[['a', 'plane', 'is', 'taking', 'off', '.'],
 ['an', 'air', 'plane', 'is', 'taking', 'off', '.'],
 ['a', 'man', 'is', 'playing', 'a', 'large', 'flute', '.'],
 ['a', 'man', 'is', 'playing', 'a', 'flute', '.'],
 ['a', 'man', 'is', 'spreading', 'shreded', 'cheese', 'on', 'a', 'pizza', '.']]

In [182]:
len(tokenized_sents)

11498

In [ ]:
# Create set of vocabulary
vocab = {token for sent in tokenized_sents for token in sent}

In [184]:
len(vocab)

12466

### Compute IDF

In [ ]:
# Number of documents
num_doc = len(tokenized_sents)

# To store IDF values
idf = {}

for token in vocab:

    # Calculate document frequency
    doc_freq = sum(1 for sent in tokenized_sents if token in sent)

    # Calculate idf for each token
    idf[token] = math.log((num_doc + 1) / (doc_freq  + 1)) # +1 is smoothing, ZeroDivisionError handle

In [186]:
idf

{'cavender': 8.65686817348872,
 'pixel-perfect': 8.251403065380556,
 'repugnant': 8.251403065380556,
 'mechanic': 8.251403065380556,
 'conditions': 7.270573812368829,
 'settled': 7.963720992928774,
 'specified': 8.65686817348872,
 'kazan': 7.55825588482061,
 'hospital': 5.145322734657698,
 'potties': 8.65686817348872,
 'spokeswoman': 7.270573812368829,
 "'are": 8.65686817348872,
 'cold': 6.952120081250294,
 'plead': 7.7405774416145645,
 'involvement': 7.55825588482061,
 'exploited': 8.65686817348872,
 'cac-40': 8.251403065380556,
 'thirty-seven': 8.65686817348872,
 'brass': 8.65686817348872,
 'go': 6.516802009992449,
 '196': 8.65686817348872,
 'southern': 6.258972900690349,
 'johnston': 7.963720992928774,
 'ten': 6.865108704260664,
 'positive': 7.404105204993352,
 'rumours': 8.65686817348872,
 'mechanisms': 8.65686817348872,
 'dignitaries': 8.251403065380556,
 'fundamental': 7.55825588482061,
 'enlist': 8.65686817348872,
 'studio': 8.251403065380556,
 'rwanda': 8.251403065380556,
 'tra

### Compute TF and TF-IDF

In [ ]:
# Define compute_tf fucntion to compute TF
def compute_tf(sent):

    # Lowercase tokens keep them in a list
    tokens= [token.lower() for token in word_tokenize(sent)]

    # Compute frequency of each word
    token_freq = FreqDist(tokens)

    tf = {} # to store tf values

    for token, freq in token_freq.items():

        # Compute TF
        tf[token] = freq / len(tokens)

    return tf

In [ ]:
# Define compute_tfidf fucntion to compute TF-IDF
def compute_tfidf(sent, idf):

    # Compute TF 
    tf = compute_tf(sent)

    tfidf = {} # To store tf-idf values
    
    for token, tf_value in tf.items():

        # Ignore OOV tokens
        if token in idf:

            # Compute TF-IDF
            tfidf[token] = tf_value * idf[token]

    return tfidf

In [ ]:
# Compute Cosine similarity between two TF-IDF vectors
def compute_cosine_similarity(vec1, vec2, vocab):

    # Convert dictionary vectors into ordered lists
    v1 = [vec1.get(token, 0) for token in vocab]
    v2 = [vec2.get(token, 0) for token in vocab]

    # Calculate the dot product of the two vectors
    dot_product = np.dot(v1, v2)

    # Calculate the magnitude of each vector
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    # Compute cosine similarity
    cos_sim = dot_product / (norm_v1 * norm_v2)

    return cos_sim

In [190]:
dataset["validation"]

Dataset({
    features: ['sentence1', 'sentence2', 'score'],
    num_rows: 1500
})

In [ ]:
validation_df = []

for sample in dataset["validation"]:
    
    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    tfidf1 = compute_tfidf(sent1, idf)
    tfidf2 = compute_tfidf(sent2, idf)

    similarity = compute_cosine_similarity(tfidf1, tfidf2, vocab)

    validation_df.append({"sentence1": sent1,
                          "sentence2": sent2,
                          "human_score": score,
                          "pred_score": similarity})
    

In [195]:
validation_df = pd.DataFrame(validation_df)

validation_df.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.894515
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.880716
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,0.973565
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.820029
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.854905


In [ ]:
pearson, _ = pearsonr(validation_df["pred_score"], validation_df["human_score"])
spearman, _ = spearmanr(validation_df["pred_score"], validation_df["human_score"])

print("Pearson:", pearson)
print("Spearman:", spearman)

Pearson: 0.7187242305725757
Spearman: 0.7177199307370453


## Manual TF-IDF without stopwords

In [198]:
stop_words = stopwords.words("english")

In [ ]:
sents_out_stopwords = []

for sent in tokenized_sents:

    # Remove stopwords and punctuations
    sent_filtered = [token for token in sent if token.isalpha() and token not in stop_words]
    
    sents_out_stopwords.append(sent_filtered)

In [ ]:
# Create new vocabulary without punctuation and stopwords
vocab_no_stopwords = {token for sent in sents_out_stopwords for token in sent}

### Compute new IDF for 'vocab_no_stopwords'

In [ ]:
num_doc2 = len(sents_out_stopwords)

idf2 = {}

for token in vocab_no_stopwords:
    
    doc_freq = sum(1 for sent in sents_out_stopwords if token in sent)

    idf2[token] = math.log((num_doc2 + 1) / (doc_freq + 1))

In [ ]:
def compute_tf2(sent):

    tokenized_sent = [token.lower() for token in word_tokenize(sent) if token.lower().isalpha() and token.lower() not in stop_words]
    token_freq = FreqDist(tokenized_sent)

    tf = {}
    for token, freq in token_freq.items():
        tf[token] = freq / len(tokenized_sent)

    return tf

In [223]:
def compute_tfidf2(sent, idf):

    tf = compute_tf2(sent)

    tfidf = {}
    
    for token, tf_value in tf.items():

        # Ignore OOV tokens
        if token in idf:
            tfidf[token] = tf_value * idf[token]

    return tfidf

In [ ]:
validation_df_no_stopwords = []

for sample in dataset["validation"]:
    
    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    tfidf1 = compute_tfidf2(sent1, idf2)
    tfidf2 = compute_tfidf2(sent2, idf2)

    similarity = compute_cosine_similarity(tfidf1, tfidf2, vocab_no_stopwords)

    validation_df_no_stopwords.append({"sentence1": sent1,
                                       "sentence2": sent2,
                                       "human_score": score,
                                       "pred_score": similarity})

In [225]:
validation_df2 = pd.DataFrame(validation_df_no_stopwords)

validation_df2.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.913711
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.871020
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.818149
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.862552


In [226]:
pearson, _ = pearsonr(validation_df2["pred_score"], validation_df2["human_score"])
spearman, _ = spearmanr(validation_df2["pred_score"], validation_df2["human_score"])

print("Pearson:", pearson)
print("Spearman:", spearman)

Pearson: 0.7067850988383765
Spearman: 0.7103656714469587


## Scikit-learn TF-IDF

In [135]:
train_sentences = []

for sample in dataset["train"]:
    train_sentences.append(sample["sentence1"])
    train_sentences.append(sample["sentence2"])

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")

vectorizer.fit(train_sentences)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [ ]:
validation_df_sci = []

for sample in dataset["validation"]:
    
    vec1 = vectorizer.transform([sample["sentence1"]])
    vec2 = vectorizer.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    validation_df_sci.append({"sentence1": sample["sentence1"],
                               "sentence2": sample["sentence2"],
                               "human_score": sample["score"],
                               "pred_score": similarity})

In [146]:
validation_df3 = pd.DataFrame(validation_df_sci)

validation_df3.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.907134
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.870008
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.787308
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.828134


In [147]:
pearson, _ = pearsonr(validation_df3["pred_score"], validation_df3["human_score"])
spearman, _ = spearmanr(validation_df3["pred_score"], validation_df3["human_score"])

print("Pearson:", pearson)
print("Spearman:", spearman)

Pearson: 0.725847933123757
Spearman: 0.7276137089272163


## TF-IDF with n-grams

In [ ]:
vectorizer2 = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))

vectorizer2.fit(train_sentences)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [ ]:
validation_df_sci2 = []

for sample in dataset["validation"]:
    
    vec1 = vectorizer2.transform([sample["sentence1"]])
    vec2 = vectorizer2.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    validation_df_sci2.append({"sentence1": sample["sentence1"],
                               "sentence2": sample["sentence2"],
                               "human_score": sample["score"],
                               "pred_score": similarity})

In [151]:
validation_df4 = pd.DataFrame(validation_df_sci2)

validation_df4.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.763102
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.745688
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.606828
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.678414


In [152]:
pearson, _ = pearsonr(validation_df4["pred_score"], validation_df4["human_score"])
spearman, _ = spearmanr(validation_df4["pred_score"], validation_df4["human_score"])

print("Pearson:", pearson)
print("Spearman:", spearman)

Pearson: 0.6992180816071115
Spearman: 0.7112181113570575


## TF-IDF with sublinear TF


In [215]:
vectorizer3 = TfidfVectorizer(lowercase=True, stop_words="english", sublinear_tf=True)

vectorizer3.fit(train_sentences)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [ ]:
validation_df_sci3 = []

for sample in dataset["validation"]:
    
    vec1 = vectorizer3.transform([sample["sentence1"]])
    vec2 = vectorizer3.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    validation_df_sci3.append({"sentence1": sample["sentence1"],
                               "sentence2": sample["sentence2"],
                               "human_score": sample["score"],
                               "pred_score": similarity})

In [217]:
validation_df5 = pd.DataFrame(validation_df_sci3)

validation_df5.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.907604
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.870042
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.787424
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.828471


In [218]:
pearson, _ = pearsonr(validation_df5["pred_score"], validation_df5["human_score"])
spearman, _ = spearmanr(validation_df5["pred_score"], validation_df5["human_score"])

print("Pearson:", pearson)
print("Spearman:", spearman)

Pearson: 0.725681920168936
Spearman: 0.7273780358638556


## Validation comparison across models

## Final test evaluation for the two strongest models

### Scikit-learn TF-IDF

In [ ]:
test_df_sci = []

for sample in dataset["test"]:
    
    vec1 = vectorizer.transform([sample["sentence1"]])
    vec2 = vectorizer.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    test_df_sci.append({"sentence1": sample["sentence1"],
                        "sentence2": sample["sentence2"],
                        "human_score": sample["score"],
                        "pred_score": similarity})

In [232]:
test_df = pd.DataFrame(test_df_sci)

test_df.head()

,sentence1,sentence2,human_score,pred_score
0,A girl is styling her hair.,A girl is brushing her hair.,0.50,0.489599
1,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.,0.72,0.616470
2,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.,1.00,0.690376
3,A man is cutting up a cucumber.,A man is slicing a cucumber.,0.84,0.698280
4,A man is playing a harp.,A man is playing a keyboard.,0.30,0.324109


In [233]:
pearson, _ = pearsonr(test_df["pred_score"], test_df["human_score"])
spearman, _ = spearmanr(test_df["pred_score"], test_df["human_score"])

print("Test Pearson:", pearson)
print("Test Spearman:", spearman)

Test Pearson: 0.6292127868191818
Test Spearman: 0.6136647523389394


### TF-IDF with sublinear TF

In [ ]:
test_df_sci2 = []

for sample in dataset["test"]:
    
    vec1 = vectorizer3.transform([sample["sentence1"]])
    vec2 = vectorizer3.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    test_df_sci2.append({"sentence1": sample["sentence1"],
                         "sentence2": sample["sentence2"],
                         "human_score": sample["score"],
                         "pred_score": similarity})

In [234]:
test_df2 = pd.DataFrame(test_df_sci2)

test_df2.head()

,sentence1,sentence2,human_score,pred_score
0,A girl is styling her hair.,A girl is brushing her hair.,0.50,0.479794
1,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.,0.72,0.616394
2,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.,1.00,0.626075
3,A man is cutting up a cucumber.,A man is slicing a cucumber.,0.84,0.700021
4,A man is playing a harp.,A man is playing a keyboard.,0.30,0.321180


In [235]:
pearson, _ = pearsonr(test_df2["pred_score"], test_df2["human_score"])
spearman, _ = spearmanr(test_df2["pred_score"], test_df2["human_score"])

print("Test Pearson:", pearson)
print("Test Spearman:", spearman)

Test Pearson: 0.6303073254472652
Test Spearman: 0.6149506682574165


## Explanation of why TF-IDF works and where it fails